# Скачивание ML и LLM кейсов

Запускайте ячейки по порядку из корня репозитория или папки `scraper`.
Колонки входного CSV: `Title`, `Link`; номер 1 — первая запись после заголовка
(многострочные поля CSV считаются одной записью).
Результаты: `scraped/0001_Title_с_подчеркиваниями.md`, лог: `scrape_log.csv`.
Markdown содержит Title, ссылку и извлеченное тело статьи либо текст PDF.

Статусы: **успешно**, **ошибка**, **пропущено**, **требует проверки**.
Подозрительные результаты сохраняются: короткий текст, признаки блокировки,
ошибки кодировки, PDF с пустыми страницами, неудачный браузерный fallback.
Эти эвристики не гарантируют полноту статьи. Сканы требуют отдельного OCR.
Видео и прочие форматы пропускаются, CAPTCHA и платный доступ не обходятся.

Сначала можно задать `LIMIT = 10`. Для полного прохода — `LIMIT = None`.
Лог дописывается после каждой ссылки; повторный запуск сохраняет историю попыток
и пропускает уже сохраненные результаты. Ошибки повторяются; для повторной
обработки подозрительных результатов установите `RETRY_REVIEW = True`.
`RESUME = False` повторяет весь проход. Одновременно запускайте только один notebook.

Ячейки установки добавляют Playwright и Chromium для браузерной попытки при HTTP 403
и страниц с JavaScript. Chromium устанавливается в ядре текущего notebook.
`USE_BROWSER = True` включен по умолчанию; `False` оставляет только HTTP.
После HTTP 403 браузер пробует открыть конечный URL один раз. Если Playwright
не установлен или браузер тоже получает отказ, причина записывается в лог.
Страницы CAPTCHA и блокировки не сохраняются как статьи.
Браузерный fallback не изменяет настройки прокси и TLS.
Для скачивания нужен сетевой доступ к доменам из CSV.


In [ ]:
%pip install "httpx>=0.27,<1" "trafilatura>=2,<3" "PyMuPDF>=1.24,<2" "playwright>=1.48,<2"


In [ ]:
import subprocess
import sys

# Requires download access; skip this cell if using USE_BROWSER = False.
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)


In [ ]:
import csv
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import httpx
import pymupdf
import trafilatura

CSV_NAME = '800 ML and LLM use cases.csv'
BASE_DIR = Path.cwd() if (Path.cwd() / CSV_NAME).exists() else Path.cwd() / 'scraper'
CSV_PATH = BASE_DIR / CSV_NAME
OUTPUT_DIR = BASE_DIR / 'scraped'
LOG_PATH = BASE_DIR / 'scrape_log.csv'
TIMEOUT = httpx.Timeout(30, connect=10)
ATTEMPTS = 3
REQUEST_PAUSE = 1.0  # Sequential processing limits load on each host.
MAX_BYTES = 30 * 1024 * 1024
MIN_CHARS = 800
LIMIT = None  # Set to 10 for a trial run; None processes every CSV row.
RESUME = True
RETRY_REVIEW = False  # True reprocesses results requiring review.
USE_BROWSER = True  # Playwright fallback for HTTP 403 and empty/short HTML results.
USER_AGENT = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
              'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36')



In [ ]:
FIELDS = ['row_number', 'title', 'url', 'status', 'error', 'output_file', 'final_url', 'characters', 'timestamp']
VIDEO_HOSTS = ('youtube.com', 'youtu.be', 'vimeo.com', 'dailymotion.com', 'rutube.ru', 'loom.com')
SKIP_EXTENSIONS = {'.mp4', '.webm', '.mov', '.mp3', '.wav', '.zip', '.gz', '.ppt', '.pptx', '.doc', '.docx', '.jpg', '.png', '.gif', '.xlsx'}

class SkipSource(Exception):
    pass

def filename(number, title):
    clean = unicodedata.normalize('NFKC', title)
    clean = re.sub(r'[\s<>:"/\\|?*\x00-\x1f]+', '_', clean).strip('._') or 'untitled'
    # Limit UTF-8 bytes as well as characters for common filesystem limits.
    clean = clean.encode('utf-8')[:180].decode('utf-8', errors='ignore').rstrip('._')
    return f'{number:04d}_{clean}.md'

def validate_url(url, title=''):
    parsed = urlparse(url)
    if parsed.scheme not in {'http', 'https'} or not parsed.hostname:
        raise ValueError('Некорректная HTTP/HTTPS ссылка')
    host = parsed.hostname.lower()
    if any(host == h or host.endswith('.' + h) for h in VIDEO_HOSTS) or '[VIDEO]' in title.upper():
        raise SkipSource('Видео')
    if Path(parsed.path.lower()).suffix in SKIP_EXTENSIONS:
        raise SkipSource('Неподдерживаемый формат файла')

def fetch(client, url):
    for attempt in range(ATTEMPTS):
        try:
            time.sleep(REQUEST_PAUSE)
            with client.stream('GET', url) as response:
                response.raise_for_status()
                validate_url(str(response.url))
                mime = response.headers.get('content-type', '').split(';')[0].lower().strip()
                if mime and mime not in {'text/html', 'application/xhtml+xml', 'application/pdf', 'application/octet-stream', 'binary/octet-stream'}:
                    raise SkipSource(f'Неподдерживаемый Content-Type: {mime}')
                chunks, size = [], 0
                started = time.monotonic()
                for chunk in response.iter_bytes():
                    size += len(chunk)
                    if size > MAX_BYTES:
                        raise ValueError(f'Документ превышает лимит {MAX_BYTES} байт')
                    if time.monotonic() - started > 120:
                        raise httpx.ReadTimeout('Превышен общий срок загрузки', request=response.request)
                    chunks.append(chunk)
                data = b''.join(chunks)
                if data.lstrip().startswith(b'%PDF-'):
                    kind = 'pdf'
                elif mime in {'text/html', 'application/xhtml+xml'} or (not mime and re.search(br'<(?:html|!doctype\s+html)', data[:1024], re.I)):
                    kind = 'html'
                elif mime == 'application/pdf':
                    raise ValueError('Сервер объявил PDF, но сигнатура PDF отсутствует')
                else:
                    raise SkipSource(f'Не HTML и не PDF: {mime or "тип не указан"}')
                return kind, data, str(response.url)
        except (httpx.TimeoutException, httpx.NetworkError, httpx.RemoteProtocolError, httpx.HTTPStatusError) as exc:
            retryable = not isinstance(exc, httpx.HTTPStatusError) or exc.response.status_code in {408, 429, 500, 502, 503, 504}
            if not retryable or attempt + 1 == ATTEMPTS:
                raise
            delay = 2 ** (attempt + 1)
            if isinstance(exc, httpx.HTTPStatusError):
                header = exc.response.headers.get('retry-after', '')
                if header.isdigit():
                    delay = min(60, max(delay, int(header)))
            time.sleep(delay)

def html_text(data):
    return trafilatura.extract(data, output_format='txt', include_comments=False,
                               include_tables=True, favor_precision=True) or ''

async def browser_text(url):
    # Notebook supports top-level await; async Playwright works with its event loop.
    from playwright.async_api import async_playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        try:
            page = await browser.new_page(user_agent=USER_AGENT)
            response = await page.goto(url, wait_until='domcontentloaded', timeout=30000)
            if response is None or response.status >= 400:
                raise ValueError(f'Ошибка браузера: HTTP {response.status if response else "нет ответа"}')
            validate_url(page.url)
            mime = response.headers.get('content-type', '').lower()
            if 'text/html' not in mime and 'application/xhtml+xml' not in mime:
                raise SkipSource('Браузер получил не HTML')
            await page.wait_for_timeout(2000)
            return html_text(await page.content()), page.url
        finally:
            await browser.close()

def pdf_text(data):
    with pymupdf.open(stream=data, filetype='pdf') as doc:
        if doc.needs_pass:
            raise ValueError('PDF защищен паролем')
        pages = [page.get_text(sort=True).strip() for page in doc]
    nonempty = [text for text in pages if text]
    warning = f'Нет текста на {len(pages) - len(nonempty)} из {len(pages)} страниц; возможно, требуется OCR' if len(nonempty) < len(pages) else ''
    return '\n\n'.join(nonempty), warning

def review_reason(text):
    reasons = []
    if len(text.strip()) < MIN_CHARS:
        reasons.append(f'Короткий текст: {len(text.strip())} символов (порог {MIN_CHARS})')
    if re.search(r'verify you are human|checking your browser|access denied|enable javascript|subscribe to continue|sign in to continue|just a moment|captcha', text, re.I):
        reasons.append('Возможна блокировка, CAPTCHA или неполный доступ')
    if '\ufffd' in text:
        reasons.append('Возможна ошибка кодировки')
    return '; '.join(reasons)

def read_rows():
    with CSV_PATH.open(encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if not {'Title', 'Link'}.issubset(reader.fieldnames or []):
            raise ValueError('CSV должен содержать колонки Title и Link')
        return [(n, row.get('Title', '').strip(), row.get('Link', '').strip()) for n, row in enumerate(reader, 1)]

def previous_results():
    if not LOG_PATH.exists():
        return {}
    with LOG_PATH.open(encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != FIELDS:
            raise ValueError('Структура существующего лога отличается; переименуйте его перед запуском')
        return {(row['row_number'], row['title'], row['url']): row for row in reader}

def append_log(result):
    new = not LOG_PATH.exists() or LOG_PATH.stat().st_size == 0
    with LOG_PATH.open('a', encoding='utf-8-sig' if new else 'utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            writer.writeheader()
        writer.writerow(result)
        f.flush()

async def run():
    rows = read_rows()
    previous = previous_results()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    counts = Counter()
    with httpx.Client(timeout=TIMEOUT, follow_redirects=True,
                      headers={'User-Agent': USER_AGENT}) as client:
        for number, title, url in rows[:LIMIT]:
            path = OUTPUT_DIR / filename(number, title)
            prior = previous.get((str(number), title, url))
            if RESUME and prior:
                if prior['status'] == 'пропущено' or (prior['status'] in ({'успешно'} if RETRY_REVIEW else {'успешно', 'требует проверки'}) and path.exists() and path.stat().st_size):
                    counts['уже обработано'] += 1
                    continue
            result = dict.fromkeys(FIELDS, '')
            result.update(row_number=number, title=title, url=url, timestamp=datetime.now(timezone.utc).isoformat())
            try:
                validate_url(url, title)
                browser_used = False
                try:
                    kind, data, final_url = fetch(client, url)
                except httpx.HTTPStatusError as exc:
                    if exc.response.status_code != 403 or not USE_BROWSER:
                        raise
                    try:
                        text, final_url = await browser_text(str(exc.response.url))
                    except Exception as browser_exc:
                        raise ValueError(f'HTTP 403; браузерная попытка не удалась: {type(browser_exc).__name__}: {browser_exc}') from browser_exc
                    kind, data, browser_used = 'html', b'', True
                result['final_url'] = final_url
                warning = ''
                if kind == 'pdf':
                    text, warning = pdf_text(data)
                else:
                    if not browser_used:
                        text = html_text(data)
                    if USE_BROWSER and not browser_used and len(text.strip()) < MIN_CHARS:
                        try:
                            rendered, rendered_url = await browser_text(final_url)
                            if len(rendered) > len(text):
                                text = rendered
                                final_url = rendered_url
                        except Exception as exc:
                            warning = f'Браузерный fallback: {type(exc).__name__}: {exc}'
                if not text.strip():
                    raise ValueError('Не удалось извлечь тело статьи или текст PDF; для сканов нужен OCR')
                if kind == 'html' and re.search(r'verify you are human|checking your browser|access denied|just a moment|captcha', text[:2000], re.I):
                    raise ValueError('Получена страница блокировки или CAPTCHA вместо статьи')
                result['final_url'] = final_url
                reason = '; '.join(filter(None, [warning, review_reason(text)]))
                if not title:
                    reason = '; '.join(filter(None, [reason, 'Title отсутствует в CSV']))
                result.update(status='требует проверки' if reason else 'успешно', error=reason, characters=len(text))
                tmp = path.with_suffix('.md.tmp')
                tmp.write_text(f'# {title}\n\nИсточник: {url}\n\n{text.strip()}\n', encoding='utf-8')
                tmp.replace(path)
                result['output_file'] = str(path.relative_to(BASE_DIR))
            except SkipSource as exc:
                result.update(status='пропущено', error=str(exc))
            except Exception as exc:
                result.update(status='ошибка', error=f'{type(exc).__name__}: {exc}')
            append_log(result)
            counts[result['status']] += 1
            print(f'{number:04d}: {result["status"]} — {title}')
    print(dict(counts))
    return counts



In [ ]:
counts = await run()
